# SatQuery — train the S1+S2 land-cover classifier

Stage 1 of the pipeline: a multi-label classifier over the 19 BigEarthNet
classes, trained on both Sentinel-1 (SAR) and Sentinel-2 (optical) patches.

**Before running:**

1. `Settings → Accelerator → GPU T4 x2` (or P100)
2. `Settings → Persistence → Files and Variables` so a disconnect does not wipe checkpoints
3. `+ Add Input` → the `bigearthnet-vqa` dataset

The training loop checkpoints every 10 minutes and auto-resumes, so if the
session dies you can just re-run the training cell.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Get the code

In [ ]:
# The training code lives in this repo. If it is private, create a GitHub
# personal access token and use:
#   REPO = "https://<TOKEN>@github.com/shubh000015/satquery-ai.git"
# Alternatively upload the ml/ folder as a private Kaggle Dataset and point
# ML_DIR at /kaggle/input/<your-dataset>/ml instead of cloning.
import subprocess, sys, os
from pathlib import Path

REPO = "https://github.com/shubh000015/satquery-ai.git"
CLONE_DIR = Path("/kaggle/working/satquery-ai")

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(CLONE_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)

ML_DIR = CLONE_DIR / "ml"
sys.path.insert(0, str(ML_DIR))
os.chdir(ML_DIR)
print("ml dir:", ML_DIR)
print(sorted(p.name for p in ML_DIR.iterdir()))

In [ ]:
# torch/torchvision/transformers are preinstalled on Kaggle; this only fills gaps.
!pip -q install "bitsandbytes>=0.45" 2>&1 | tail -2

## 2. Find the dataset

In [ ]:
# Locate the attached dataset. Add knayamket/bigearthnet-vqa via
# "+ Add Input" in the notebook sidebar first.
from pathlib import Path

CANDIDATES = sorted(Path("/kaggle/input").glob("*/train.jsonl"))
if not CANDIDATES:
    raise SystemExit(
        "No train.jsonl under /kaggle/input. Attach the bigearthnet-vqa dataset "
        "with '+ Add Input'."
    )

DATA_DIR = CANDIDATES[0].parent
print("dataset:", DATA_DIR)
for sub in ("images_s2", "images_s1"):
    folder = DATA_DIR / sub
    print(f"  {sub}: {'present' if folder.exists() else 'MISSING'}")

## 3. Smoke test

400 patches, one epoch. This proves the data loads, labels parse, and the loss
falls before you spend hours on it.

In [ ]:
import os
os.environ["DATA_DIR"] = str(DATA_DIR)   # so the shell cells below can see it

!python train_classifier.py --data $DATA_DIR --out /kaggle/working/smoke \
    --limit 400 --epochs 1 --batch-size 8 --workers 2

## 4. The real run

Writes to `/kaggle/working/landcover`. Roughly 40 minutes per epoch on a T4 at
~120k patches — adjust `--epochs` to the session time you have. Re-running this
cell after a disconnect resumes from the last checkpoint.

In [ ]:
!python train_classifier.py --data $DATA_DIR --out /kaggle/working/landcover \
    --epochs 6 --batch-size 64 --workers 2 --save-every-minutes 10

## 5. Score it on the held-out split

In [ ]:
!python evaluate_classifier.py --data $DATA_DIR \
    --checkpoint /kaggle/working/landcover/classifier.pt \
    --out /kaggle/working/landcover/eval_report.json

In [ ]:
import json
report = json.load(open("/kaggle/working/landcover/eval_report.json"))
print(json.dumps(report["overall"], indent=2)[:1200])
print()
print("by modality:", json.dumps(report["byModality"], indent=2)[:800])

## 6. End-to-end check

Run one patch through both stages. `enabled=False` on the verbalizer keeps this
fast — set it to `True` to pull Qwen2.5 and see the phrased answer.

In [ ]:
from PIL import Image
from satquery_ml.inference import LandCoverPredictor
from satquery_ml.verbalizer import Verbalizer, resolve_facts

predictor = LandCoverPredictor("/kaggle/working/landcover/classifier.pt")
verbalizer = Verbalizer(enabled=False)   # True -> loads Qwen2.5-7B-Instruct

sample = sorted((DATA_DIR / "images_s2").glob("*.png"))[0]
image = Image.open(sample)
prediction = predictor.predict(image, modality="optical")

print("labels:", prediction.short_labels())
print("top scores:", [(n, round(s, 3)) for n, s in prediction.top(5)])
print()
for question in [
    "Is there any water in this image?",
    "Which land-cover classes are present?",
    "Is there urban fabric here?",
]:
    facts = resolve_facts(prediction, question)
    print(f"Q: {question}")
    print(f"   one-word: {facts.one_word}  (confidence {facts.confidence})")
    print(f"   answer  : {verbalizer.phrase(facts)}")

## 7. Keep the weights

`classifier.pt` is only ~100 MB. Two ways to keep it:

- **Download it** from the Output tab and serve it from a machine with a GPU.
- **Save this notebook's output as a Dataset**, then attach that dataset to
  `kaggle_serve_tunnel.ipynb` to serve it from Kaggle directly.

In [ ]:
!ls -lh /kaggle/working/landcover